In [53]:
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM as lstm
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from keras import Input
import pmdarima as pm
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from sklearn.linear_model import LinearRegression
from datasets import load_dataset

# Modèles

## Persistence

In [369]:
def persistence(train, test=False): # pas d'ensemble de validation
    prediction = train.iloc[-1]
    if type(test) != bool:
        # Évaluation du modèle
        n_test = len(test)
        predictions = prediction * np.ones(n_test)
        rmse = np.sqrt(mean_squared_error(test, predictions))
        mae = mean_absolute_error(test, predictions)
        return {'prediction':prediction, 'rmse':rmse, 'mae':mae}
    return {'prediction':prediction}

## ARIMA

In [364]:
def ARIMA(train, saisonalite, test=False, type_produit=False): # pas d'ensemble de validation
    if type_produit:
        min_train = train.min()
        train = train - train.min() + 1  # décale tout pour que min > 0
        train = np.log(train)
    # fit
    fit_arima = pm.auto_arima(train, seasonal=saisonalite)

    if type_produit:
        train = np.exp(train) + min_train - 1 # on remet le train comme avant
    
    if type(test) != bool:
        # Évaluation du modèle
        n_test = len(test)
        predictions = fit_arima.predict(n_periods=n_test)

        # retour à l'échelle originale
        if type_produit:
            predictions = np.exp(predictions) + min_train - 1
        
        rmse_arima = np.sqrt(mean_squared_error(test, predictions))
        mae_arima = mean_absolute_error(test, predictions)
        
        return {'modele_fit':fit_arima, 'rmse':rmse_arima, 'mae':mae_arima}
    return {'modele_fit':fit_arima}

## STL + ARIMA sur le résidu

In [375]:
def STL_ARIMA(train, saisonalite, s_window, test=False, type_produit=False): # s_window : largeur de la fenetre saisonniere lors du lissage
    # local pour déterminer la saisonalité
    
    # Transformation log si multiplicatif
    if type_produit:
        min_train = train.min()
        train = train - train.min() + 1  # décale tout pour que min > 0
        train = np.log(train)
    
    # décomposition STL
    stl = STL(train, period=saisonalite, seasonal=s_window)
    
    if type_produit:
        train = np.exp(train) + min_train - 1 # on remet le train comme avant
    
    stl_result = stl.fit() # trouve la décomposition tendance, saisonalité, et bruit
    
    trend = stl_result.trend
    seasonal = stl_result.seasonal
    resid = stl_result.resid

    # extrapolation de la tendance
    t = np.arange(len(trend)).reshape(-1,1)
    model_trend = LinearRegression().fit(t, trend)

    # saisonnalité
    season_pattern = seasonal.iloc[-saisonalite:].values
    
    # fit ARIMA sur les résidus
    fit_resid = pm.auto_arima(resid, seasonal=False)
    
    if type(test) != bool:
        n_test = len(test)

        # Prévision des résidus
        pred_resid = fit_resid.predict(n_periods=n_test)
    
        # tendance future
        t_future = np.arange(len(trend), len(trend)+n_test).reshape(-1,1)
        trend_future = model_trend.predict(t_future)
    
        # répétition de la saisonnalité 
        season_future = np.tile(season_pattern, int(np.ceil(n_test/saisonalite)))[:n_test] # tile répète le motif saisonnier
    
        # reconstruction
        y_pred = trend_future + season_future + pred_resid
    
        # Retour échelle originale
        if type_produit:
            y_pred = np.exp(y_pred)
    
        rmse = np.sqrt(mean_squared_error(test, y_pred))
        mae = mean_absolute_error(test, y_pred)
        return {'model_trend':model_trend, 'season_pattern':season_pattern, 'resid_fit':fit_resid, 'rmse':rmse, 'mae':mae}
    return {'model_trend':model_trend, 'season_pattern':season_pattern, 'resid_fit':fit_resid}

## SARIMAX

In [376]:
def SARIMAX_model(y_train, X_train, saisonalite, test_y=False, test_X=False, type_produit=False):
    
    if type_produit:
        min_train = y_train.min()
        y_train = y_train - min_train + 1  # décale tout pour que min > 0
        y_train = np.log(y_train)

    # Entraînement du modèle SARIMAX
    fit_sarimax = pm.auto_arima(
        y=y_train,
        X=X_train,       # la variable cible ne dépend pas seulement de son passé, mais aussi d’autres signaux mesurés par le procédé
        seasonal=saisonalite,
        suppress_warnings=True,   # Cela cache certains warnings produits pendant la recherche du meilleur modèle
        error_action="ignore",    # Si une combinaison de paramètres provoque une erreur, elle est ignorée
        stepwise=True             # Le modèle cherche à trouver de bons paramètres plus intelligemment et plus rapidement 
    )

    if type_produit:
        y_train = np.exp(y_train) + min_train - 1 # on remet le train comme avant
    
    if type(test) != bool:
        # Prédictions
        n_test = len(test_y)   # On veut prédire exactement autant de périodes que la taille du test
        predictions = fit_sarimax.predict(
            n_periods=n_test,        # nombre de périodes futures à prédire.
            X=test_X      # Comme le modèle a été entraîné avec des variables exogènes, il a besoin des valeurs de ces variables sur la période future
        )
    
        # Retour à l'échelle originale si log appliqué
        if type_produit:
            predictions = np.exp(predictions)
    
        # Évaluation
        rmse = np.sqrt(mean_squared_error(test_y, predictions))
        mae = mean_absolute_error(test_y, predictions)
        return {'modele_fit':fit_sarimax, 'rmse':rmse, 'mae':mae}

    return {'modele_fit':fit_sarimax}

## STL + ARIMAX

In [377]:
def SARIMAX_STL_model(y_train, X_train, saisonalite, s_window, test_y=False, test_X=False, type_produit=False):
    
    # log si multiplicatif
    if type_produit:
        min_train = y_train.min()
        y_train = y_train - min_train + 1
        y_train = np.log(y_train)
    
    # Décomposition STL si demandé
    stl = STL(train, period=saisonalite, seasonal=s_window)
    stl_result = stl.fit()
    trend = stl_result.trend
    seasonal = stl_result.seasonal
    resid = stl_result.resid

    if type_produit:
        y_train = np.exp(y_train) + min_train - 1 # on remet le train comme avant
    
    # Fit SARIMAX sur la série ou sur les résidus
    fit_sarimax = pm.auto_arima(
        y=resid,
        X=X_train,
        seasonal=False,       # si tu veux ajouter saisonnalité via STL
        suppress_warnings=True,
        error_action="ignore",
        stepwise=True
    )


    # tendance future
    t = np.arange(len(trend)).reshape(-1,1)
    model_trend = LinearRegression().fit(t, trend)
    n_trend = len(trend)

    # saisonnalité répétée
    season_pattern = seasonal.iloc[-saisonalite:].values
    
    # Prédiction si test fourni
    if type(test) != bool:
        n_test = len(test_y)
        pred_resid = fit_sarimax.predict(n_periods=n_test, X=test_X)
        
        # tendance future
        t_future = np.arange(n_trend, n_trend+n_test).reshape(-1,1)
        trend_future = model_trend.predict(t_future)

        # saisonnalité répétée
        season_future = np.tile(season_pattern, int(np.ceil(n_test/saisonalite)))[:n_test]
        
        # reconstruction
        y_pred = pred_resid + trend_future + season_future
        
        # retour à l'échelle originale
        if type_produit:
            y_pred = np.exp(y_pred) + min_train - 1
        
        rmse = np.sqrt(mean_squared_error(test_y, y_pred))
        mae = mean_absolute_error(test_y, y_pred)
        return {'model_trend':model_trend, 'season_pattern':season_pattern, 'resid_fit':fit_sarimax, 'rmse':rmse, 'mae':mae}
    
    return {'model_trend':model_trend, 'season_pattern':season_pattern, 'resid_fit':fit_sarimax}

## LSTM

In [74]:
def create_sequences(data, taille_fenêtre):
    
    X = []
    y = []

    # on parcourt la série
    for i in range(len(data) - taille_fenêtre):
        
        # séquence d'entrée
        X.append(data[i:i+taille_fenêtre])
        
        # valeur à prédire
        y.append(data[i+taille_fenêtre])

    return np.array(X), np.array(y)

In [378]:
def LSTM(train, val, saisonalite, test=False, n_neurones = 32, optimizer="adam", loss="mse", metrics=["mae"], epochs=80, batch_size=16, patience=10):
    # test = False si on ne veut pas tester le modèle sur un ensmble de test

    # Le scaler transforme les données pour qu'elles soient entre 0 et 1.
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    # On apprend le minimum et le maximum à partir du train seulement.
    # Cela évite d'utiliser des informations du futur.
    
    # Transformer en 2D pour sklearn
    train_scaled = scaler.fit_transform(train.values.reshape(-1,1))
    val_scaled   = scaler.transform(val.values.reshape(-1,1))
    
    if test is not None:
        test_scaled = scaler.transform(test.values.reshape(-1,1))

    # Création des séquences

    L = saisonalite

    # création des séquences pour train
    X_train, y_train = create_sequences(train_scaled, L)
    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
    X_val, y_val = create_sequences(val_scaled, L)
    X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))

    if type(test) != bool:
        # création des séquences pour test
        X_test, y_test = create_sequences(test_scaled, L)
        X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    # Les données sont transformées en séquences afin de permettre au LSTM
    # d'apprendre les dépendances temporelles.
    # 
    # On apprend au modèle : “Si les 12 derniers mois ressemblent à ceci → le mois suivant devrait être cela.”
    
    # Chaque entrée du modèle correspond à une fenêtre temporelle
    # contenant les L observations précédentes.
    #
    # La sortie correspond à la valeur suivante de la série.
    #
    # Cette transformation permet de reformuler le problème de prévision
    # en un problème supervisé adapté aux réseaux de neurones.

    # Construction du modèle LSTM

    # Création d'un modèle séquentiel
    model = Sequential()
    
    # Couche LSTM
    # 32 neurones = taille de la mémoire du réseau
    model = Sequential([
    Input(shape=(X_train.shape[1], 1)),
    lstm(n_neurones),
    Dense(1)
    ])

    model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=metrics
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True
    )

    model.fit(
    X_train,
    y_train,
    epochs=epochs,  # nombre d’itérations d’apprentissage
    batch_size=batch_size,    # nombre d'exemples traités à la fois
    validation_data=(X_val,y_val),
    callbacks=[early_stop],
    shuffle=False # série temporelle
    )

    if type(test) != bool:
        # Évaluation du modèle
        predictions = model.predict(X_test)
    
        # Remettre les valeurs à l’échelle originale; Comme on a normalisé les données, il faut annuler la normalisation.
        predictions = scaler.inverse_transform(predictions)
        y_test_real = scaler.inverse_transform(y_test.reshape(-1,1))
    
        rmse_lstm = np.sqrt(mean_squared_error(y_test_real, predictions))
        mae_lstm = mean_absolute_error(y_test_real, predictions)
        
        return {'modele_fit':model, 'rmse':rmse_lstm, 'mae':mae_lstm, 'scaler':scaler}
    return {'modele_fit':model, 'scaler':scaler}
    

## LSTM multi-pas direct

In [76]:
def create_sequences_multi(data, window_size, horizon):

    X = []
    y = []

    for i in range(len(data) - window_size - horizon + 1):

        X.append(data[i:i+window_size])
        y.append(data[i+window_size:i+window_size+horizon])

    return np.array(X), np.array(y)

In [379]:
def LSTM_multi_pas_direct(train, val, saisonalite, test=False, taille_fenetre=12, n_pas = 12, n_neurones = 32, optimizer="adam", 
                   loss="mse", metrics=["mae"], n_epochs=80, batch_size=16, patience=10):
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    # On apprend le minimum et le maximum à partir du train seulement.
    # Cela évite d'utiliser des informations du futur.
    
    # Transformer en 2D pour sklearn
    train_scaled = scaler.fit_transform(train.values.reshape(-1,1))
    val_scaled   = scaler.transform(val.values.reshape(-1,1))


    if len(val_scaled) < taille_fenetre + n_pas:
        print('erreur, il faut un ensemble de validation plus grand !')
        return(None)

    # Création des séquences

    L = saisonalite
    X_train_multi, y_train_multi = create_sequences_multi(train_scaled, L, n_pas)
    X_train_multi = X_train_multi.reshape((X_train_multi.shape[0], X_train_multi.shape[1], 1))
    X_val_multi, y_val_multi = create_sequences_multi(val_scaled, L, n_pas)
    X_val_multi = X_val_multi.reshape((X_val_multi.shape[0], X_val_multi.shape[1], 1))

    if type(test) != bool:
        # On applique exactement la même transformation au jeu de test.
        test_scaled = scaler.transform(test.values.reshape(-1,1))
        if len(test_scaled) < taille_fenetre + n_pas:
            print('erreur, il faut un ensemble de test plus grand !')
            return(None)
        
        X_test_multi, y_test_multi = create_sequences_multi(test_scaled, L, n_pas)
        X_test_multi = X_test_multi.reshape((X_test_multi.shape[0], X_test_multi.shape[1], 1))

    # MODELE DIRECT MULTI-PAS

    model = Sequential([
    Input(shape=(taille_fenetre, 1)),
    lstm(n_neurones),
    Dense(n_pas) # multipas direct
    ])
    
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=metrics
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True
    )

    
    model.fit(
        X_train_multi,
        y_train_multi,
        epochs=n_epochs,
        batch_size=batch_size,
        validation_data=(X_val_multi, y_val_multi),
        shuffle=False,
        callbacks=[early_stop]
    )

    if type(test) != bool:
        # Évaluation du modèle
        predictions = model.predict(X_test_multi)
    
        # Remettre les valeurs à l’échelle originale; Comme on a normalisé les données, il faut annuler la normalisation.
        predictions = scaler.inverse_transform(predictions.reshape(-1,1)).reshape(predictions.shape)
        y_test_real = scaler.inverse_transform(y_test_multi.reshape(-1,1)).reshape(y_test_multi.shape)
    
        rmse_lstm = np.sqrt(mean_squared_error(y_test_real.flatten(), predictions.flatten()))
        mae_lstm = mean_absolute_error(y_test_real.flatten(), predictions.flatten())
        
        return {'modele_fit':model, 'rmse':rmse_lstm, 'mae':mae_lstm, 'scaler':scaler}
    
    return {'modele_fit':model, 'scaler':scaler}    

## ETS/Holt-Winters

In [380]:
def ETS(train, saisonalite, test=False, type_produit=False): # freq='MS' : données mensuelles au début du mois (Month Start)
    if test is not False:
        test = test.reindex(pd.date_range(start=test.index.min(), end=test.index.max(), freq='MS')).interpolate()

    
    # train doit avoir un index datetime
    if not isinstance(train.index, pd.DatetimeIndex):
        raise ValueError("train doit avoir un DatetimeIndex")

    # Transformation log si multiplicatif
    if type_produit:
        min_train = train.min()
        train = train - min_train + 1
        train = np.log(train)
    
    hw_model = ExponentialSmoothing(
        train,
        trend="add",
        seasonal="add",
        seasonal_periods=saisonalite
    ).fit()

    if type_produit:
        train = np.exp(train) + min_train - 1 # on remet le train comme avant

    if type(test) != bool:
        # Évaluation du modèle
        hw_pred = hw_model.forecast(len(test))

        # retour à l'échelle originale si log
        if type_produit:
            hw_pred = np.exp(hw_pred) + min_train - 1

        rmse_hw = np.sqrt(mean_squared_error(test, hw_pred))
        mae_hw = mean_absolute_error(test, hw_pred)
        
        return {'modele_fit':hw_model, 'rmse':rmse_hw, 'mae':mae_hw}
    
    return {'modele_fit':hw_model}

# Exemple d'utilisation

In [444]:
# Exemple : dataset tourisme (fictive local)

dataset = load_dataset("zaai-ai/time_series_dataset_residuals")
# Choisir le split 'train' (ou 'test')
train_split = dataset['train']

# Convertir ce split en DataFrame
df = train_split.to_pandas()

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df = df.set_index('Date')
df = df.groupby("Date").mean(numeric_only=True) # on aggrege les doublons

# Target
y = df['residuals'][:1000]

# proportions
train_frac = 0.6
val_frac = 0.2
test_frac = 0.2

n_total = len(y)
# print(n_total) # DEBUG

# indices de split
train_end = int(n_total * train_frac)
val_end   = int(n_total * (train_frac + val_frac))

# Splits
train = y.iloc[:train_end]
val   = y.iloc[train_end:val_end]
test  = y.iloc[val_end:]

# on normalise après le split pour éviter le leakage
mean_train = train.mean()
std_train = train.std()

train = (train - mean_train) / std_train
val   = (val - mean_train) / std_train
test  = (test - mean_train) / std_train

def regularize(series):
    return series.reindex(
        pd.date_range(series.index.min(), series.index.max(), freq="MS")
    ).interpolate()

print(df.index.duplicated().sum())

train = regularize(train)
val = regularize(val)
test = regularize(test)

train_val = pd.concat([train, val]).sort_index()

# # supprimer les doublons (si présents)
# # Vérifie les doublons
# print(train.index.duplicated().sum())
# print(val.index.duplicated().sum())
# print(train_val.index.duplicated().sum())
# print(test.index.duplicated().sum())
# train = train[~train.index.duplicated(keep='first')]
# val = val[~val.index.duplicated(keep='first')]
# train_val = train_val[~train_val.index.duplicated(keep='first')]
# test = test[~test.index.duplicated(keep='first')]

# print(train.index.is_monotonic_increasing)
# print(val.index.is_monotonic_increasing)
# print(train_val.index.is_monotonic_increasing)
# print(test.index.is_monotonic_increasing)
# # on trie par date et on réindexe en attribuant une fréquence

print(train.isna().sum())
print(val.isna().sum())
print(train_val.isna().sum())
print(test.isna().sum())

0
0
0
0
0


## Persistence

In [412]:
resultat = persistence(train_val, test)

In [413]:
print(resultat['rmse'], resultat['mae'])
resultat['prediction'] * np.ones(10)

1.3264159931223503 1.2471888088229721


array([-0.82926123, -0.82926123, -0.82926123, -0.82926123, -0.82926123,
       -0.82926123, -0.82926123, -0.82926123, -0.82926123, -0.82926123])

## ARIMA

In [414]:
resultat = ARIMA(train_val, 12, test=test, type_produit=True) # 12 mois par an

In [415]:
print(resultat['rmse'], resultat['mae'])
predictions = np.exp(resultat['modele_fit'].predict(n_periods = len(test))) + train.min() - 1
predictions[:10]

0.6552700417215283 0.5495346819604401


2014-03-01    0.422415
2014-04-01    0.287981
2014-05-01   -0.341811
2014-06-01   -0.250442
2014-07-01    0.011228
2014-08-01   -0.249247
2014-09-01    0.009910
2014-10-01   -0.248063
2014-11-01    0.008605
2014-12-01   -0.246890
Freq: MS, dtype: float64

## STL + ARIMA

In [416]:
resultat = STL_ARIMA(train_val, 12, 13, test=test, type_produit=True)

In [417]:
print(resultat['rmse'], resultat['mae'])

2.008000022098207 1.9767699524617137


In [418]:
def forecast_stl_arima(model_trend, season_pattern, min_train, fit_resid, n_train, n_test, saisonalite, type_produit=False):

    # Prévision des résidus
    pred_resid = fit_resid.predict(n_periods=n_test)

    # tendance
    t_future = np.arange(n_train, n_train+n_test).reshape(-1,1)
    trend_future = model_trend.predict(t_future)

    # répétition de la saisonnalité
    season_future = np.tile(season_pattern, int(np.ceil(n_test/saisonalite)))[:n_test] # tile répète le motif saisonnier

    # reconstruction
    y_pred = trend_future + season_future + pred_resid

    # Retour échelle originale
    if type_produit:
        y_pred = np.exp(y_pred) + min_train - 1 # on remet le train comme avant

    return y_pred

In [419]:
predictions = forecast_stl_arima(resultat['model_trend'], resultat['season_pattern'], train.min(), 
                                 resultat['resid_fit'], len(train_val), len(test), 12, type_produit=True)
predictions[:10]

2014-03-01   -0.446249
2014-04-01    0.466825
2014-05-01   -0.556019
2014-06-01   -0.778957
2014-07-01    0.322194
2014-08-01   -0.570562
2014-09-01   -0.709039
2014-10-01    0.445238
2014-11-01   -0.081954
2014-12-01   -0.471017
Freq: MS, dtype: float64

## SARIMAX

### Nouvelles données (besoin de covariables)

In [420]:
# Covariables dynamiques
X = df[['CPI', 'Inflation_Rate', 'GDP']][:1000]

# On standardise
X = (X - X.mean()) / X.std()

# Train
X_train = X.iloc[:train_end].astype(float)

# Validation
X_val = X.iloc[train_end:val_end].astype(float)

# Test
X_test = X.iloc[val_end:].astype(float)

# Train-Val
X_train_val = pd.concat([X_train, X_val])

# on standardise y
y = (y - y.mean()) / y.std()

# création des ensembles
train = y.iloc[:train_end]
val   = y.iloc[train_end:val_end]
test  = y.iloc[val_end:]

train_val = pd.concat([train, val])

In [424]:
# entraînement + évaluation
resultat = SARIMAX_model(
    train_val,
    X_train_val,
    saisonalite=12,
    test_y = test,
    test_X = X_test,
    type_produit=True
)

print(resultat['rmse'], resultat['mae'])

1.8315167508514358 1.754514964080148


In [426]:
predictions = resultat['modele_fit'].predict(n_periods=len(test),        # nombre de périodes futures à prédire.
            X=X_test)      # Comme le modèle a été entraîné avec des variables exogènes, il a besoin des valeurs de ces variables sur la période future
y_pred = np.exp(predictions) + train_val.min() - 1
y_pred[:10]

2014-03-01   -0.685694
2014-04-01    0.044934
2014-05-01   -0.981832
2014-06-01   -0.636513
2014-07-01   -0.387335
2014-08-01   -0.856426
2014-09-01   -0.614421
2014-10-01   -0.549726
2014-11-01   -0.766171
2014-12-01   -0.617912
Freq: MS, dtype: float64

## STL + ARIMAX

In [427]:
# entraînement + évaluation

resultat = SARIMAX_STL_model(train, X_train, 12, 13, test_y=test, test_X=X_test, type_produit=True)

print(resultat['rmse'], resultat['mae'])

2.87136101591047 2.797466561860805


In [430]:
def forecast_stl_sarimax(model_trend, season_pattern, min_train, fit_resid, n_train, n_test, saisonalite, X_test, type_produit=False):

    # Prévision des résidus
    pred_resid = fit_resid.predict(n_periods=n_test, X=X_test)

    # tendance
    t_future = np.arange(n_train, n_train+n_test).reshape(-1,1)
    trend_future = model_trend.predict(t_future)

    # répétition de la saisonnalité
    season_future = np.tile(season_pattern, int(np.ceil(n_test/saisonalite)))[:n_test] # tile répète le motif saisonnier

    # reconstruction
    y_pred = trend_future + season_future + pred_resid

    # Retour échelle originale
    if type_produit:
        y_pred = np.exp(y_pred) + min_train - 1 # on remet le train comme avant

    return y_pred

In [431]:
predictions = forecast_stl_sarimax(resultat['model_trend'], resultat['season_pattern'], train.min(), 
                                 resultat['resid_fit'], len(train_val), len(test), 12, X_test, type_produit=True)
predictions[:10]

2011-05-01   -2.703618
2011-06-01   -2.615274
2011-07-01   -2.383311
2011-08-01   -2.592673
2011-09-01   -2.497749
2011-10-01   -2.101638
2011-11-01   -2.459715
2011-12-01   -2.740884
2012-01-01   -2.334685
2012-02-01   -2.826327
Freq: MS, dtype: float64

## LSTM

### Simple

In [434]:
resultat_simple = LSTM(train, val, 12, test=test) # 12 mois par an

Epoch 1/80
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 94ms/step - loss: 0.0925 - mae: 0.2537 - val_loss: 0.0569 - val_mae: 0.2073
Epoch 2/80
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0546 - mae: 0.1691 - val_loss: 0.0275 - val_mae: 0.1228
Epoch 3/80
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0324 - mae: 0.1185 - val_loss: 0.0147 - val_mae: 0.0959
Epoch 4/80
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0267 - mae: 0.1148 - val_loss: 0.0161 - val_mae: 0.1138
Epoch 5/80
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0281 - mae: 0.1307 - val_loss: 0.0155 - val_mae: 0.1108
Epoch 6/80
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0259 - mae: 0.1206 - val_loss: 0.0143 - val_mae: 0.0979
Epoch 7/80
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0251 - mae: 0.1110 - val_loss: 0.0149 - val_mae: 0.0957
Epoch 8/80
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0256 - mae: 0.1089 - val_loss: 0.0150 - val_mae: 0.0956
Epoch 9/80
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 0.0257 - mae: 0.1087 - 

In [435]:
print(resultat_simple['rmse'], resultat_simple['mae'])

0.5090347227763783 0.44075243709144446


### Multi-pas direct

In [436]:
resultat_direct = LSTM_multi_pas_direct(train, val, 12, test=test) 

Epoch 1/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 106ms/step - loss: 0.1015 - mae: 0.2776 - val_loss: 0.0949 - val_mae: 0.2845
Epoch 2/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.0890 - mae: 0.2548 - val_loss: 0.0830 - val_mae: 0.2625
Epoch 3/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0783 - mae: 0.2331 - val_loss: 0.0720 - val_mae: 0.2398
Epoch 4/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0682 - mae: 0.2107 - val_loss: 0.0612 - val_mae: 0.2150
Epoch 5/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0582 - mae: 0.1872 - val_loss: 0.0500 - val_mae: 0.1871
Epoch 6/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0482 - mae: 0.1629 - val_loss: 0.0385 - val_mae: 0.1573
Epoch 7/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.0387 - mae: 0.1408 - val_loss: 0.0277 - val_mae: 0.1296
Epoch 8/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0315 - mae: 0.1265 - val_loss: 0.0203 - val_mae: 0.1119
Epoch 9/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.0275 - mae: 0.1222 -

In [437]:
print(resultat_direct['rmse'], resultat_direct['mae'])

0.6486133156413636 0.5381347251638628


### Prévision multi-pas LSTM récursif

In [438]:
def forecast_lstm(model, val, scaler, saisonalite, n_steps):

    # scaler comme dans l'entraînement
    val_scaled = scaler.transform(val.values.reshape(-1, 1))

    # dernière fenêtre
    last_window = val_scaled[-saisonalite:]

    preds = []

    for _ in range(n_steps):

        X = last_window.reshape(1, saisonalite, 1)

        pred = model.predict(X, verbose=0)

        preds.append(pred[0,0])

        # mise à jour de la fenêtre
        last_window = np.vstack([last_window[1:], pred])

    preds = np.array(preds).reshape(-1,1)

    # retour à l'échelle originale
    preds = scaler.inverse_transform(preds)

    return preds

In [440]:
predictions = forecast_lstm(
    resultat_simple['modele_fit'],
    val,
    resultat_simple['scaler'],
    saisonalite=12,
    n_steps=10 # prédit les 10 prochaines périodes
)
predictions

array([[-0.27286738],
       [-0.32255092],
       [-0.6099452 ],
       [-0.49887922],
       [-0.2309122 ],
       [-0.30603623],
       [ 0.02422912],
       [ 0.22736838],
       [-0.33105487],
       [-0.17227991]], dtype=float32)

### Prévision multi-pas LSTM direct

In [441]:
def forecast_lstm_direct(model, val, scaler, saisonalite):
    # scaler comme dans ton entraînement
    val_scaled = scaler.transform(val.values.reshape(-1, 1))

    # dernière fenêtre
    last_window = val_scaled[-saisonalite:]

    # Mise en forme pour LSTM
    X = last_window.reshape(1, saisonalite, 1)

    # Prédiction directe
    preds = model.predict(X)

    # Retour à l’échelle originale
    preds = scaler.inverse_transform(preds.reshape(-1,1)).reshape(preds.shape)

    return preds.flatten()

In [442]:
predictions = forecast_lstm_direct(
    resultat_direct['modele_fit'],   # modèle direct multi-step
    val,
    resultat_direct['scaler'],
    saisonalite=12
)
predictions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step


array([-0.17609608, -0.09165937, -0.0016542 , -0.11644023, -0.02399205,
       -0.17829502, -0.2296937 , -0.20548749, -0.1279988 , -0.23446819,
       -0.24114737, -0.22027726], dtype=float32)

## ETS/Holt-Winter

In [348]:
# supprimer les doublons (si présents)
# Vérifie les doublons
print(train_val.index.duplicated().sum())
print(test.index.duplicated().sum())
train_val = train_val[~train_val.index.duplicated(keep='first')]
test = test[~test.index.duplicated(keep='first')]

630
30


In [349]:
print(test.index.is_monotonic_increasing)
print(test.index.is_monotonic_increasing)
# on trie par date et on réindexe en attribuant une fréquence

train_val = train_val.sort_index()
train_val = train_val.reindex(pd.date_range(start=test.index.min(), end=test.index.max(), freq='MS')).interpolate()

test = test.sort_index()
test = test.reindex(pd.date_range(start=test.index.min(), end=test.index.max(), freq='MS')).interpolate()

False
False


In [445]:
resultat = ETS(train_val, 12, type_produit=True, test=test)

In [447]:
print(resultat['rmse'], resultat['mae'])
predictions = np.exp(resultat['modele_fit'].forecast(len(test))) + train.min() - 1
predictions[:10]

0.6498440791305248 0.5409209441100593


2014-03-01   -0.088035
2014-04-01    0.521389
2014-05-01   -0.348290
2014-06-01   -0.423010
2014-07-01    0.396822
2014-08-01   -0.425874
2014-09-01   -0.175507
2014-10-01    0.561710
2014-11-01   -0.231733
2014-12-01   -0.254643
Freq: MS, dtype: float64